<div style="
    background: linear-gradient(135deg, #8B1E3F 0%, #C44569 55%, #F8A5C2 100%);
    padding: 50px 35px;
    border-radius: 22px;
    color: white;
    text-align: center;
    box-shadow: 0 8px 25px rgba(139, 30, 63, 0.25);
">

<div style="font-size: 14px; letter-spacing: 3px; text-transform: uppercase; opacity: 0.9; margin-bottom: 15px;">
    BinX Tech — AI & ML Internship · Week 6 · Day 1
</div>

<h1 style="font-size: 38px; margin: 0; font-weight: 700;">
    Sprint 1 Planning & Baseline
</h1>

<p style="font-size: 20px; margin-top: 12px; opacity: 0.95;">
    Cardiac Patient Monitoring System — Heart Disease Prediction
</p>

</div>

## Sprint 1 Plan

**Duration:** Week 6 — 5 days  
**Project:** Cardiac Patient Monitoring System  
**Dataset:** Heart Disease Prediction (`heart.csv` — 918 observations)

---

###  Sprint Goal

Build and evaluate a neural network for heart disease prediction that is
trained, tuned, and compared against the Random Forest baseline established
in the previous phase.

---

### Product Backlog

| # | Task | Deliverable | Day |
|---|------|-------------|-----|
| 1 | Sprint planning + baseline recording | This notebook | Day 1 |
| 2 | Neural network foundations | `day2_nn_foundations.ipynb` | Day 2 |
| 3 | Training mechanics | `day3_training_mechanics.ipynb` | Day 3 |
| 4 | Keras network — build, compile, train | `day4_keras_network.ipynb` | Day 4 |
| 5 | Tuning + Sprint Review + Retrospective | `day5_tuning_sprint_review.ipynb` | Day 5 |

---

###  Definition of Done

- All five notebooks run without errors
- Neural network trained and evaluated on the test set
- Results compared to the RF baseline in a metric table
- All work committed to GitHub with clear commit messages
- Sprint Retrospective written at the end of Day 5

---

## Step 1 — Import Libraries

The cell below loads all libraries needed for this notebook:
data handling, visualization, the preprocessing pipeline,
and the Random Forest classifier used to record the baseline.

In [30]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report
)

SEED = 42
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded successfully ✓")

Libraries loaded successfully ✓


---

## Step 2 — Load Data & Split

I load `heart.csv` and apply the same 60/20/20 stratified split used in
the cardiac monitoring project. The split is done before any preprocessing
to prevent data leakage.

In [31]:
# Load dataset
df = pd.read_csv("E:\BinX_AI_ML_Internship\Cardiac_Project\Data\heart.csv")

print(f"Dataset shape: {df.shape}")
print(f"\nTarget distribution:\n{df['HeartDisease'].value_counts()}")
print(f"\nClass balance: {df['HeartDisease'].mean():.1%} positive (heart disease)")

Dataset shape: (918, 12)

Target distribution:
HeartDisease
1    508
0    410
Name: count, dtype: int64

Class balance: 55.3% positive (heart disease)


In [32]:
# Define features and target
X = df.drop(columns=["HeartDisease"])
y = df["HeartDisease"]

# Step 1: 60% train / 40% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=SEED
)

# Step 2: 50% of temp → validation / 50% → test (= 20% each)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
)

print(f"Train size : {X_train.shape[0]} rows")
print(f"Val size   : {X_val.shape[0]} rows")
print(f"Test size  : {X_test.shape[0]} rows")

Train size : 550 rows
Val size   : 184 rows
Test size  : 184 rows


---

## Step 3 — Preprocessing Pipeline

I build the same preprocessing pipeline from the cardiac monitoring project:
- **Numeric features:** median imputation → standard scaling
- **Binary features:** passed through as-is
- **Categorical features:** most-frequent imputation → one-hot encoding

In [33]:
# Define feature groups
numeric_features     = ["Age", "RestingBP", "Cholesterol", "MaxHR", "Oldpeak"]
binary_features      = ["FastingBS"]
categorical_features = ["Sex", "ChestPainType", "RestingECG",
                        "ExerciseAngina", "ST_Slope"]

# Numeric pipeline
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler())
])

# Categorical pipeline
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# Full preprocessor
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline,     numeric_features),
    ("bin", "passthrough",        binary_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Preprocessing pipeline defined ✓")

Preprocessing pipeline defined ✓


---

## Step 4 — Baseline Model (Random Forest)

I train the tuned Random Forest from the cardiac monitoring project and
record its validation and test scores. These numbers are the target
the neural network needs to beat in Sprint 1.

In [34]:
# Build full pipeline: preprocessor + RF with tuned hyperparameters
baseline_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=SEED
    ))
])

# Train on training set only
baseline_pipeline.fit(X_train, y_train)

# Predict on validation and test sets
y_val_pred  = baseline_pipeline.predict(X_val)
y_test_pred = baseline_pipeline.predict(X_test)

y_val_proba  = baseline_pipeline.predict_proba(X_val)[:, 1]
y_test_proba = baseline_pipeline.predict_proba(X_test)[:, 1]

print("Baseline model trained ✓")

Baseline model trained ✓


In [35]:
# Record baseline metrics
baseline_results = {
    "Accuracy"  : [accuracy_score(y_val,  y_val_pred),
                   accuracy_score(y_test, y_test_pred)],
    "Precision" : [precision_score(y_val,  y_val_pred),
                   precision_score(y_test, y_test_pred)],
    "Recall"    : [recall_score(y_val,  y_val_pred),
                   recall_score(y_test, y_test_pred)],
    "F1-score"  : [f1_score(y_val,  y_val_pred),
                   f1_score(y_test, y_test_pred)],
    "AUC-ROC"   : [roc_auc_score(y_val,  y_val_proba),
                   roc_auc_score(y_test, y_test_proba)]
}

baseline_df = pd.DataFrame(
    baseline_results,
    index=["Validation", "Test"]
).round(4)

print("=" * 50)
print("       SPRINT 1 BASELINE — Random Forest")
print("=" * 50)
print(baseline_df.to_string())
print("=" * 50)

       SPRINT 1 BASELINE — Random Forest
            Accuracy  Precision  Recall  F1-score  AUC-ROC
Validation    0.9130     0.9388  0.9020    0.9200   0.9440
Test          0.8859     0.8716  0.9314    0.9005   0.9427


---

## Step 4 — Baseline Results Interpretation

The table above shows the Random Forest performance on both the validation
and test sets.

**Validation Set**
- Accuracy of 0.913 — the model classified 91.3% of patients correctly.
- F1-score of 0.920 and AUC-ROC of 0.944 — strong overall performance
  with good balance between precision and recall.

**Test Set**
- Accuracy drops slightly to 0.886, which is expected when moving from
  validation to unseen data.
- Recall of 0.931 is the most important number here — it means the model
  correctly identified 93.1% of patients who actually have heart disease.
  In a medical screening context, missing a sick patient (false negative)
  is more costly than a false alarm.
- AUC-ROC of 0.943 confirms the model separates the two classes well
  across all decision thresholds.

**Sprint 1 Target**
The neural network built in Days 2–4 needs to match or exceed these numbers
to justify the added complexity — particularly the test F1-score of 0.900
and AUC-ROC of 0.943.

---

## Step 5 — Sprint 1 Summary

### Baseline Recorded ✓

| Metric | Validation | Test |
|--------|------------|------|
| Accuracy | 0.9130 | 0.8859 |
| Precision | 0.9388 | 0.8716 |
| Recall | 0.9020 | 0.9314 |
| F1-score | 0.9200 | 0.9005 |
| AUC-ROC | 0.9440 | 0.9427 |

### What Comes Next

| Day | Task |
|-----|------|
| Day 2 | Neural network foundations — neurons, layers, activation functions |
| Day 3 | Training mechanics — loss, backpropagation, optimizers |
| Day 4 | Build and train a Keras network on this dataset |
| Day 5 | Tune the network and compare results to the baseline above |

### Key Reminder
The test set is now closed — it will not be touched again until the
final evaluation in Day 5. All tuning and architecture decisions in
Days 2–4 will be based on the validation set only.